In [1]:
import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)


In [2]:
from analyse.advertising.e01_rupture_detection.rupture_detector import (
    RuptureDetector,
    print_summary,
)
from dateutil import tz
from datetime import datetime
from analyse.advertising.tools.testimony_table.extract import get_testimony_data
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)

tz_paris = tz.gettz("Europe/Paris")
start_time = datetime(2025, 3, 11, 12, 00, 0, tzinfo=tz_paris)
end_time = datetime(2025, 3, 11, 13, 0, 0, tzinfo=tz_paris)

CHANNEL = "fr3-idf"
TESTIMONY_CHANNEL = "FRANCE 3"

annotations = get_testimony_data(
    channel=TESTIMONY_CHANNEL,
    from_date=start_time,
    to_date=end_time,
    source_file="export.csv",
)

detector = RuptureDetector(
    sensitivity=0.1,
    context_sec=1.0,  # durée analysée de chaque côté d'un point pour évaluer la rupture.
    max_ruptures=0,  # 0 pour pas de limite
    sr=22050,  # Fréquence d'échantillonnage de travail
    hop_length=512,  # Pas entre frames (~23ms à 22050Hz)
    n_mfcc=10,  # Nombre de coefficients MFCC
    novelty_smooth_sec=0.5,  # Lissage temporel de la courbe de nouveauté.
    min_segment_sec=1.0,  # Durée minimale d'un segment pour être considéré comme tel.
    silence_percentile=8.0,  # Percentile pour définir le seuil de silence
    cosine_weight=1.0,
)

len(annotations)

2

In [3]:
from analyse.advertising.e00_download.mediatree import CachedMediatreeAPI

mediatree_api = CachedMediatreeAPI()
input_file = mediatree_api.download_export(CHANNEL, start_time, end_time)
witness_file = mediatree_api.api.get_single_export_url(CHANNEL, start_time, end_time, media_format="mp4")

# input_file = "/root/Workspace/quotaclimat/analyse/advertising/exports/tf1_2025-03-04_12-00-00_2025-03-04_13-00-00.mp3"
# witness_file = (
#     "https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4"
# )

In [4]:
segments, peaks, novelty, features, y = detector.run(input_file)

print_summary(segments)

generate_player(
    media_input=witness_file,
    segments=[s.to_dict() for s in segments],
    annotations=format_annotation(annotations, from_date=start_time),
    output_path="output.html",
    params=detector.get_params(),
    novelty_peaks=detector.get_novelty_peaks(novelty),
    start_epoch=int(start_time.timestamp()),
)

[1/5] Chargement : ../exports/fr3-idf_2025-03-11_11-00-00Z_2025-03-11_12-00-00Z.mp3
      Durée : 3600.0s  (60.0 min)
      → 9.91s
[2/5] Extraction des features...
      → 8.18s
[3/5] Calcul de la courbe de nouveauté...
      Fenêtre de contexte : 43 frames = 1.00s de chaque côté
      Seuil silence       : énergie < 0.00627 (percentile 8.0) → 24821 frames silencieuses (16.0%)
      Lissage             : 21 frames = 0.49s
      → 3.54s
[4/5] Détection des frontières...
      Seuil sensitivity=0.10 → hauteur min 0.0068 (percentile 90)
      Après seuil + distance min (1.0s) : 539 ruptures candidates
      Résultat final : 539 ruptures → 540 segments
      → 0.00s
[5/5] Segmentation terminée : 540 segments  → 0.02s
      Durée totale : 21.65s

═══════════════════════════════════════════════════════
  RÉSUMÉ DE SEGMENTATION
═══════════════════════════════════════════════════════
  Total segments : 540
  Label                     Nb       %   Durée moy
  ──────────────────────────────────